# Agentic Graph RAG — Giai đoạn 1: Baseline RAG với LlamaIndex

Notebook này dựng nhánh **LlamaIndex RAG** trong kiến trúc Hybrid RAG. Kèm theo là phần dùng chung **Context + Sources → LLM → Answer + Citations**, để GraphRAG và Query Router ở các giai đoạn sau cắm vào.

```
User → Query Router ─┬─ câu hỏi cụ thể   → LlamaIndex RAG    ← giai đoạn 1 (notebook này)
                     ├─ câu hỏi quan hệ  → GraphRAG Local     ← giai đoạn 2
                     └─ câu hỏi tổng hợp → GraphRAG Global    ← giai đoạn 2
                                  ↓
           Context + Sources → LLM → Answer + Citations       ← giai đoạn 1 (dùng chung)
```

| Mục | Nội dung |
|---|---|
| 1–2 | Cài thư viện; cấu hình LLM và embedding |
| 3 | Tạo tài liệu mẫu (giả lập), metadata đặt trong front matter |
| 4 | Đọc tài liệu, chunk theo cấu trúc Chương/Điều |
| 5 | Embedding rồi lưu vào Qdrant (local mode) |
| 6 | Truy xuất có lọc theo sản phẩm, sinh câu trả lời kèm trích dẫn |
| 7 | Đánh giá nhanh trên bộ câu hỏi mẫu |

> **Lưu ý về dữ liệu:** tài liệu trong notebook là **dữ liệu giả lập**, không phải sản phẩm thật. Nếu dùng LLM API bên ngoài (Gemini/OpenAI), nội dung các chunk sẽ được gửi ra ngoài. Trước khi đưa tài liệu nội bộ thật vào, cần xác nhận việc này được phép; nếu không, đặt `LLM_PROVIDER = "openai_like"` và trỏ tới model tự host.

## 1. Cài đặt thư viện

In [ ]:
%pip install -q llama-index-core llama-index-readers-file llama-index-vector-stores-qdrant qdrant-client llama-index-embeddings-huggingface llama-index-llms-google-genai llama-index-llms-openai llama-index-llms-openai-like

## 2. Cấu hình

Mọi tham số nằm trong cell này. API key được đọc từ biến môi trường, nếu chưa có thì nhập qua ô nhập ẩn (`getpass`). **Không ghi key vào notebook.**

In [ ]:
from pathlib import Path

# LLM: "gemini" | "openai" | "openai_like" (vLLM / Ollama / LM Studio tự host, API tương thích OpenAI)
LLM_PROVIDER = "gemini"
LLM_MODELS = {
    "gemini": "gemini-2.5-flash",
    "openai": "gpt-4.1-mini",
    "openai_like": "Qwen/Qwen2.5-7B-Instruct",
}
LLM_API_BASE = "http://localhost:8000/v1"  # chỉ dùng cho openai_like
LLM_TEMPERATURE = 0.1

# Embedding chạy local: đa ngôn ngữ, tốt cho tiếng Việt (~2.3 GB, nên bật GPU)
EMBED_MODEL = "BAAI/bge-m3"

# Chunking & truy xuất
CHUNK_SIZE = 512     # token — chỉ cắt thêm khi một Điều dài hơn mức này
CHUNK_OVERLAP = 64
TOP_K = 5

# Đường dẫn — trên Colab là /content/..., sẽ mất khi runtime bị reset
BASE_DIR = Path.cwd()
DOCS_DIR = BASE_DIR / "data" / "docs"
QDRANT_PATH = BASE_DIR / "storage" / "qdrant"
REBUILD_INDEX = False  # đặt True khi đã sửa nội dung tài liệu

DOCS_DIR.mkdir(parents=True, exist_ok=True)
QDRANT_PATH.mkdir(parents=True, exist_ok=True)
# Mỗi embedding model một collection, tránh lệch số chiều vector khi đổi model
COLLECTION = "insurance_docs__" + EMBED_MODEL.split("/")[-1].lower().replace("-", "_")

print("Docs  :", DOCS_DIR)
print("Qdrant:", QDRANT_PATH, "| collection:", COLLECTION)

In [ ]:
import os
from getpass import getpass

from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


def get_secret(name: str) -> str:
    """Lấy key từ biến môi trường; chưa có thì hỏi qua ô nhập ẩn (chỉ giữ trong phiên hiện tại)."""
    if not os.environ.get(name):
        os.environ[name] = getpass(f"Nhập {name}: ")
    return os.environ[name]


def build_llm(provider: str, model: str):
    if provider == "gemini":
        from llama_index.llms.google_genai import GoogleGenAI
        return GoogleGenAI(model=model, api_key=get_secret("GOOGLE_API_KEY"), temperature=LLM_TEMPERATURE)
    if provider == "openai":
        from llama_index.llms.openai import OpenAI
        return OpenAI(model=model, api_key=get_secret("OPENAI_API_KEY"), temperature=LLM_TEMPERATURE)
    if provider == "openai_like":
        from llama_index.llms.openai_like import OpenAILike
        return OpenAILike(
            model=model,
            api_base=LLM_API_BASE,
            api_key=os.environ.get("LLM_API_KEY", "not-needed"),
            is_chat_model=True,
            context_window=32768,
            temperature=LLM_TEMPERATURE,
        )
    raise ValueError(f"LLM_PROVIDER không hợp lệ: {provider}")


llm = build_llm(LLM_PROVIDER, LLM_MODELS[LLM_PROVIDER])
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL)
Settings.llm = llm
Settings.embed_model = embed_model

print("LLM      :", LLM_PROVIDER, "/", LLM_MODELS[LLM_PROVIDER], "->", llm.complete("Chỉ trả lời đúng một từ: OK").text.strip())
print("Embedding:", EMBED_MODEL, "| dim =", len(embed_model.get_text_embedding("kiểm tra")))

## 3. Tài liệu mẫu (giả lập)

Có 3 tài liệu: **Quy tắc chung** áp dụng cho mọi sản phẩm, **Sản phẩm A** (nằm viện) và **Sản phẩm B** (bệnh hiểm nghèo). Mỗi file Markdown có **front matter** chứa metadata:

- `product_code`: dùng để lọc khi truy xuất. Giá trị `ALL` nghĩa là áp dụng cho mọi sản phẩm.
- `aliases`: các cách gọi tên sản phẩm, dùng để nhận diện sản phẩm trong câu hỏi.

Các điều kiện từ chối chi trả được cố ý đặt rải rác ở nhiều Điều và nhiều file, để làm phép thử cho câu hỏi quan hệ (GraphRAG ở giai đoạn 2).

**Khi dùng tài liệu thật:** bỏ qua 3 cell `%%writefile` bên dưới và chép file vào `data/docs/`. Markdown có front matter cho kết quả tốt nhất. PDF, DOCX và TXT vẫn đọc được nhưng sẽ thiếu metadata sản phẩm.

In [ ]:
%%writefile data/docs/quy_tac_chung_suc_khoe.md
---
product_code: ALL
product_name: Quy tắc chung bảo hiểm sức khỏe bổ trợ
doc_type: quy_tac_chung
version: "2024.1"
effective_date: "2024-01-01"
note: Tài liệu mẫu giả lập, chỉ dùng thử nghiệm
---
# Quy tắc chung – Bảo hiểm sức khỏe bổ trợ

Quy tắc này áp dụng chung cho tất cả sản phẩm bảo hiểm sức khỏe bổ trợ của Công ty. Nếu điều khoản riêng của từng sản phẩm có quy định khác thì điều khoản riêng được ưu tiên áp dụng.

## Chương I. Định nghĩa

### Điều 1. Giải thích thuật ngữ

1. **Nằm viện**: việc Người được bảo hiểm được Bệnh viện tiếp nhận điều trị nội trú liên tục tối thiểu 24 giờ theo chỉ định của bác sĩ. Điều trị trong ngày, điều trị ngoại trú hoặc lưu theo dõi tại phòng cấp cứu dưới 24 giờ không được xem là Nằm viện.
2. **Bệnh viện**: cơ sở khám chữa bệnh được cấp phép hoạt động hợp pháp tại Việt Nam và có khoa điều trị nội trú. Không bao gồm viện dưỡng lão, trung tâm điều dưỡng, cơ sở cai nghiện, cơ sở phục hồi chức năng hoặc spa.
3. **Bệnh có sẵn**: bệnh hoặc thương tật đã được chẩn đoán, đã có triệu chứng hoặc đã được điều trị trong vòng 3 năm trước Ngày hiệu lực hợp đồng.
4. **Bệnh đặc biệt**: ung thư, u lành tính, bệnh tim mạch, đột quỵ, suy thận, bệnh gan mạn tính, đái tháo đường và các biến chứng của các bệnh này.
5. **Tai nạn**: sự kiện bất ngờ, do tác động của một lực từ bên ngoài, ngoài ý muốn của Người được bảo hiểm và là nguyên nhân trực tiếp gây ra thương tật.
6. **Thời gian chờ**: khoảng thời gian tính từ Ngày hiệu lực hợp đồng mà các sự kiện bảo hiểm phát sinh trong đó không được chi trả, trừ trường hợp do Tai nạn.

## Chương II. Thời gian chờ và điều kiện chi trả

### Điều 2. Thời gian chờ

| Trường hợp | Thời gian chờ |
|---|---|
| Tai nạn | Không áp dụng |
| Bệnh thông thường | 30 ngày |
| Bệnh đặc biệt | 90 ngày |
| Bệnh có sẵn (đã kê khai và được Công ty chấp nhận) | 365 ngày |
| Biến chứng thai sản | 270 ngày |

Nếu hợp đồng mất hiệu lực và được khôi phục, Thời gian chờ được tính lại từ ngày khôi phục hiệu lực.

### Điều 3. Điều kiện chung để được chi trả

Công ty chi trả quyền lợi bảo hiểm khi thỏa mãn đồng thời các điều kiện sau:

1. Hợp đồng còn hiệu lực tại thời điểm xảy ra sự kiện bảo hiểm và phí bảo hiểm đã được đóng đầy đủ, kể cả trong thời gian gia hạn đóng phí 60 ngày.
2. Sự kiện bảo hiểm xảy ra sau khi kết thúc Thời gian chờ quy định tại Điều 2.
3. Việc điều trị là cần thiết về mặt y khoa và theo chỉ định của bác sĩ.
4. Sự kiện bảo hiểm không thuộc các trường hợp loại trừ tại Điều 4 và các điều khoản loại trừ riêng của từng sản phẩm.
5. Hồ sơ yêu cầu chi trả được nộp đầy đủ và đúng thời hạn theo Điều 6.

## Chương III. Loại trừ và từ chối chi trả

### Điều 4. Các trường hợp loại trừ chung

Công ty không chi trả bất kỳ quyền lợi nào nếu sự kiện bảo hiểm phát sinh từ:

1. Hành vi tự tử hoặc cố ý tự gây thương tích trong vòng 2 năm kể từ Ngày hiệu lực hợp đồng hoặc ngày khôi phục hiệu lực gần nhất.
2. Sử dụng rượu, bia với nồng độ cồn trong máu hoặc hơi thở vượt mức cho phép theo quy định pháp luật; sử dụng ma túy hoặc chất kích thích không theo chỉ định của bác sĩ.
3. Hành vi vi phạm pháp luật hình sự hoặc tham gia đánh nhau, trừ trường hợp tự vệ chính đáng.
4. Chiến tranh, bạo loạn, khủng bố, nhiễm phóng xạ.
5. Phẫu thuật hoặc điều trị thẩm mỹ, chỉnh hình răng, điều trị vô sinh, kiểm tra sức khỏe định kỳ, tiêm chủng.
6. Bệnh có sẵn không được kê khai trong Hồ sơ yêu cầu bảo hiểm.
7. Tham gia các môn thể thao nguy hiểm như đua xe, leo núi chuyên nghiệp, nhảy dù, lặn sâu trên 30 mét.

### Điều 5. Từ chối chi trả do vi phạm nghĩa vụ kê khai

1. Nếu Bên mua bảo hiểm hoặc Người được bảo hiểm cố ý kê khai không trung thực, hoặc không kê khai thông tin sức khỏe có ảnh hưởng đến quyết định chấp nhận bảo hiểm, Công ty có quyền từ chối chi trả và đơn phương chấm dứt hợp đồng.
2. Trường hợp vi phạm không cố ý, Công ty có thể điều chỉnh phí bảo hiểm hoặc áp dụng thêm điều khoản loại trừ thay vì chấm dứt hợp đồng.
3. Công ty từ chối chi trả đối với hồ sơ có dấu hiệu gian lận hoặc chứng từ y tế bị giả mạo, sửa chữa.

## Chương IV. Thủ tục giải quyết quyền lợi

### Điều 6. Hồ sơ và thời hạn yêu cầu chi trả

1. Thời hạn nộp hồ sơ là 12 tháng kể từ ngày xuất viện hoặc ngày kết thúc điều trị. Hồ sơ nộp quá thời hạn sẽ bị từ chối, trừ trường hợp bất khả kháng có chứng minh.
2. Hồ sơ gồm: Phiếu yêu cầu giải quyết quyền lợi; giấy ra viện; bảng kê chi tiết viện phí và hóa đơn hợp lệ; kết quả xét nghiệm, chẩn đoán hình ảnh liên quan; giấy chứng nhận phẫu thuật (nếu có); biên bản tai nạn (nếu sự kiện do tai nạn).
3. Công ty có thể yêu cầu Người được bảo hiểm giám định y khoa tại cơ sở do Công ty chỉ định; chi phí giám định do Công ty chịu.

### Điều 7. Thời hạn giải quyết

Công ty thông báo kết quả giải quyết trong vòng 15 ngày làm việc kể từ ngày nhận đủ hồ sơ hợp lệ. Trường hợp từ chối chi trả, Công ty thông báo bằng văn bản và nêu rõ lý do.

In [ ]:
%%writefile data/docs/san_pham_A_an_tam_noi_tru.md
---
product_code: A
product_name: An Tâm Nội Trú
aliases: [sản phẩm a, an tâm nội trú]
doc_type: quy_tac_san_pham
version: "2024.1"
effective_date: "2024-01-01"
note: Tài liệu mẫu giả lập, chỉ dùng thử nghiệm
---
# Sản phẩm A – Bảo hiểm An Tâm Nội Trú

Sản phẩm bổ trợ chi trả trợ cấp và chi phí khi Người được bảo hiểm phải nằm viện do bệnh hoặc tai nạn. Sản phẩm áp dụng cùng Quy tắc chung – Bảo hiểm sức khỏe bổ trợ.

## Chương I. Thông tin chung

### Điều 1. Đối tượng, thời hạn và chương trình bảo hiểm

1. Người được bảo hiểm từ 30 ngày tuổi đến 65 tuổi khi tham gia; được tái tục đến 75 tuổi.
2. Thời hạn bảo hiểm 1 năm, tái tục hằng năm.
3. Có hai chương trình: Cơ bản và Nâng cao. Chương trình được chọn ghi trên Giấy chứng nhận bảo hiểm.

## Chương II. Quyền lợi bảo hiểm

### Điều 2. Quyền lợi trợ cấp nằm viện

1. Công ty chi trả trợ cấp cho mỗi ngày Nằm viện do bệnh hoặc tai nạn:
   - Chương trình Cơ bản: 1.000.000 đồng/ngày.
   - Chương trình Nâng cao: 2.000.000 đồng/ngày.
2. Tối đa 60 ngày Nằm viện mỗi năm hợp đồng.
3. Nằm viện tại khoa Hồi sức tích cực (ICU): chi trả 200% mức trợ cấp ngày, tối đa 15 ngày ICU mỗi năm hợp đồng. Số ngày ICU được tính vào giới hạn 60 ngày.
4. Điều kiện: thời gian nằm viện tối thiểu 24 giờ liên tục theo định nghĩa Nằm viện tại Quy tắc chung.

### Điều 3. Quyền lợi phẫu thuật

Chi trả chi phí phẫu thuật thực tế, tối đa:

- Chương trình Cơ bản: 20.000.000 đồng/lần và 40.000.000 đồng/năm hợp đồng.
- Chương trình Nâng cao: 50.000.000 đồng/lần và 100.000.000 đồng/năm hợp đồng.

### Điều 4. Quyền lợi chăm sóc sau xuất viện

Chi trả một lần 2.000.000 đồng (Cơ bản) hoặc 5.000.000 đồng (Nâng cao) cho mỗi đợt Nằm viện liên tục từ 5 ngày trở lên.

## Chương III. Loại trừ riêng

### Điều 5. Các trường hợp không chi trả quyền lợi nằm viện

Ngoài các trường hợp loại trừ chung tại Điều 4 Quy tắc chung, Công ty không chi trả quyền lợi của sản phẩm này đối với:

1. Nằm viện để điều dưỡng, an dưỡng, phục hồi chức năng, hoặc chỉ để làm xét nghiệm, chẩn đoán mà không có điều trị.
2. Nằm viện do rối loạn tâm thần, trầm cảm hoặc rối loạn hành vi.
3. Nằm viện để sinh con thông thường; biến chứng thai sản phát sinh trong 270 ngày đầu kể từ Ngày hiệu lực.
4. Nằm viện tại cơ sở y tế ngoài lãnh thổ Việt Nam, trừ chương trình Nâng cao (được áp dụng tại các nước Đông Nam Á).
5. Nằm viện để điều trị HIV/AIDS và các bệnh lây truyền qua đường tình dục.

In [ ]:
%%writefile data/docs/san_pham_B_khoe_tron_doi.md
---
product_code: B
product_name: Khỏe Trọn Đời
aliases: [sản phẩm b, khỏe trọn đời]
doc_type: quy_tac_san_pham
version: "2023.2"
effective_date: "2023-07-01"
note: Tài liệu mẫu giả lập, chỉ dùng thử nghiệm
---
# Sản phẩm B – Bảo hiểm bệnh hiểm nghèo Khỏe Trọn Đời

Sản phẩm bổ trợ chi trả khi Người được bảo hiểm được chẩn đoán mắc bệnh hiểm nghèo. Sản phẩm áp dụng cùng Quy tắc chung – Bảo hiểm sức khỏe bổ trợ.

## Chương I. Thông tin chung

### Điều 1. Đối tượng và thời hạn bảo hiểm

1. Người được bảo hiểm từ 18 tuổi đến 60 tuổi khi tham gia.
2. Thời hạn bảo hiểm đến năm Người được bảo hiểm 80 tuổi; phí bảo hiểm cố định trong suốt thời hạn.
3. Số tiền bảo hiểm từ 100.000.000 đồng đến 2.000.000.000 đồng.

## Chương II. Quyền lợi bảo hiểm

### Điều 2. Quyền lợi bệnh hiểm nghèo

1. Bệnh hiểm nghèo giai đoạn đầu: chi trả 25% Số tiền bảo hiểm, tối đa 500.000.000 đồng; chi trả một lần duy nhất.
2. Bệnh hiểm nghèo giai đoạn sau: chi trả 100% Số tiền bảo hiểm trừ đi khoản đã chi trả cho giai đoạn đầu (nếu có). Hợp đồng chấm dứt sau khi chi trả quyền lợi này.
3. Danh mục bệnh hiểm nghèo được quy định tại Phụ lục 1.

### Điều 3. Quyền lợi trợ cấp nằm viện do bệnh hiểm nghèo

1. Chi trả 500.000 đồng cho mỗi ngày Nằm viện để điều trị bệnh hiểm nghèo đã được Công ty chấp nhận chi trả theo Điều 2.
2. Tối đa 30 ngày mỗi năm hợp đồng và 180 ngày trong suốt thời hạn hợp đồng.
3. Không chi trả trợ cấp nằm viện cho các bệnh không thuộc danh mục bệnh hiểm nghèo.

### Điều 4. Quyền lợi miễn đóng phí

Sau khi Công ty chi trả quyền lợi bệnh hiểm nghèo giai đoạn đầu, các kỳ phí tiếp theo của sản phẩm này được miễn và hợp đồng vẫn tiếp tục hiệu lực.

## Chương III. Loại trừ riêng

### Điều 5. Các trường hợp không chi trả

Ngoài các trường hợp loại trừ chung tại Điều 4 Quy tắc chung, Công ty không chi trả khi:

1. Bệnh hiểm nghèo được chẩn đoán hoặc có triệu chứng trong 90 ngày đầu kể từ Ngày hiệu lực (Thời gian chờ bệnh đặc biệt). Trường hợp này Công ty hoàn lại phí đã đóng của sản phẩm, không có lãi.
2. Bệnh hiểm nghèo là Bệnh có sẵn không được kê khai.
3. Ung thư biểu mô tại chỗ hoặc khối u có tính chất ác tính thấp chỉ được xem xét chi trả ở giai đoạn đầu, không chi trả ở giai đoạn sau.

## Phụ lục 1. Danh mục bệnh hiểm nghèo (rút gọn)

Ung thư; nhồi máu cơ tim; đột quỵ; suy thận giai đoạn cuối; ghép tạng chính; xơ gan giai đoạn cuối; bệnh Parkinson; bệnh Alzheimer; hôn mê trên 96 giờ; mất thị lực hoàn toàn hai mắt; bỏng nặng độ III trên 20% diện tích cơ thể.

## 4. Đọc tài liệu và chunk theo cấu trúc

- File Markdown: front matter được tách thành metadata, phần còn lại là nội dung.
- `MarkdownNodeParser` tách theo heading, mỗi Điều thành một node. Đường dẫn Chương › Điều được lưu vào metadata `section` để làm trích dẫn.
- Điều nào dài hơn `CHUNK_SIZE` thì `SentenceSplitter` cắt tiếp.
- Khi embedding, mỗi chunk chỉ kèm metadata `product_code`, `product_name` và `section`. Nhờ vậy vector biết đoạn văn thuộc sản phẩm nào, Điều nào, trong khi các metadata nhiễu như tên file hay ngày tháng bị loại ra.

In [ ]:
import re
import unicodedata

import yaml
from llama_index.core import Document, SimpleDirectoryReader

FRONT_MATTER_RE = re.compile(r"\A---\s*\n(.*?)\n---\s*\n", re.DOTALL)
OTHER_EXTS = {".pdf", ".docx", ".txt"}


def normalize(text: str) -> str:
    """Chuyển về chữ thường và bỏ dấu tiếng Việt, để 'san pham a' khớp với 'Sản phẩm A'."""
    text = text.lower().replace("đ", "d")
    return "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn")


def load_markdown(path: Path) -> tuple[Document, list[str]]:
    raw = path.read_text(encoding="utf-8")
    meta, body = {}, raw
    if m := FRONT_MATTER_RE.match(raw):
        meta = yaml.safe_load(m.group(1)) or {}
        body = raw[m.end():]
    aliases = meta.pop("aliases", None) or []
    metadata = {k: str(v) for k, v in meta.items()}
    metadata.setdefault("product_code", "ALL")
    metadata["file_name"] = path.name
    return Document(id_=path.name, text=body.strip(), metadata=metadata), aliases


def load_documents(docs_dir: Path) -> tuple[list[Document], dict[str, list[str]]]:
    documents, product_aliases = [], {}
    for path in sorted(docs_dir.glob("*.md")):
        doc, aliases = load_markdown(path)
        documents.append(doc)
        code = doc.metadata["product_code"]
        if code != "ALL":
            names = [f"san pham {code}", doc.metadata.get("product_name", ""), *aliases]
            product_aliases[code] = sorted({normalize(n) for n in names if n})

    other_files = [str(p) for p in sorted(docs_dir.iterdir()) if p.suffix.lower() in OTHER_EXTS]
    if other_files:
        for doc in SimpleDirectoryReader(input_files=other_files).load_data():
            doc.metadata.setdefault("product_code", "ALL")
            documents.append(doc)
    return documents, product_aliases


documents, PRODUCT_ALIASES = load_documents(DOCS_DIR)
print(f"{len(documents)} tài liệu:")
for doc in documents:
    print(f"  - {doc.metadata['file_name']:<36} product={doc.metadata['product_code']:<4} {len(doc.text):>6} ký tự")
print("Alias sản phẩm:", PRODUCT_ALIASES)

In [ ]:
from llama_index.core.node_parser import MarkdownNodeParser, SentenceSplitter
from llama_index.core.schema import MetadataMode

EMBED_METADATA_KEYS = {"product_code", "product_name", "section"}
MIN_BODY_CHARS = 30  # bỏ các node chỉ có tiêu đề, vd "## Chương I. Định nghĩa"


def section_of(node) -> str:
    """Ghép các header cha (header_path) với heading của node: 'Chương II › Điều 2. ...'."""
    parts = [p for p in node.metadata.get("header_path", "").split("/") if p]
    first_line = node.text.lstrip().split("\n", 1)[0]
    if first_line.startswith("#"):
        heading = first_line.lstrip("#").strip()
        if not parts or parts[-1] != heading:
            parts.append(heading)
    return " › ".join(parts)


def body_length(node) -> int:
    return sum(len(line.strip()) for line in node.text.splitlines() if not line.lstrip().startswith("#"))


def chunk_documents(documents: list[Document]):
    md_nodes = [n for n in MarkdownNodeParser().get_nodes_from_documents(documents) if body_length(n) >= MIN_BODY_CHARS]
    for n in md_nodes:
        n.metadata["section"] = section_of(n)
        n.metadata.pop("header_path", None)

    splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    nodes = splitter.get_nodes_from_documents(md_nodes)
    for n in nodes:
        n.excluded_embed_metadata_keys = [k for k in n.metadata if k not in EMBED_METADATA_KEYS]
        n.excluded_llm_metadata_keys = list(n.excluded_embed_metadata_keys)
    return nodes


nodes = chunk_documents(documents)
print(f"{len(nodes)} chunk")
for n in nodes:
    print(f"  [{n.metadata['product_code']:<3}] {len(n.text):>5} ký tự | {n.metadata['section'][:100]}")

print("\nVí dụ nội dung đưa vào embedding:\n")
print(nodes[min(1, len(nodes) - 1)].get_content(metadata_mode=MetadataMode.EMBED)[:600])

## 5. Index vào Qdrant

Qdrant chạy ở **local mode**: dữ liệu lưu thành file, không cần server. Khi lên production chỉ cần đổi `QdrantClient(path=...)` thành `QdrantClient(url=...)`.

Nếu collection đã tồn tại và số vector khớp với số chunk, notebook nạp lại index mà không embedding lại. Index được tạo lại khi số chunk thay đổi hoặc khi `REBUILD_INDEX = True`.

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

# Local mode khoá thư mục lưu trữ: phải đóng client cũ khi chạy lại cell này
if "qdrant_client" in globals():
    qdrant_client.close()
qdrant_client = QdrantClient(path=str(QDRANT_PATH))

exists = qdrant_client.collection_exists(COLLECTION)
if exists and (REBUILD_INDEX or qdrant_client.count(COLLECTION).count != len(nodes)):
    qdrant_client.delete_collection(COLLECTION)
    exists = False

vector_store = QdrantVectorStore(client=qdrant_client, collection_name=COLLECTION)
if exists:
    index = VectorStoreIndex.from_vector_store(vector_store)
    print(f"Nạp index có sẵn: {qdrant_client.count(COLLECTION).count} vector")
else:
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex(nodes, storage_context=storage_context, show_progress=True)
    print(f"Đã index: {qdrant_client.count(COLLECTION).count} vector")

## 6. Truy xuất và sinh câu trả lời kèm trích dẫn

Luồng xử lý gồm 4 bước:

1. Nhận diện sản phẩm được nhắc trong câu hỏi, có hoặc không dấu đều được.
2. Lọc theo `product_code`: chỉ lấy sản phẩm đó và quy tắc chung (`ALL`).
3. Lấy top-k chunk, đánh số thành context `[1]..[k]`.
4. LLM trả lời và trích dẫn `[n]`.

`retrieve()` và `generate()` được tách riêng có chủ đích. Ở giai đoạn 3, `retrieve()` sẽ được thay bằng router (LlamaIndex / GraphRAG Local / GraphRAG Global), còn **format `sources`** và `generate()` giữ nguyên. Kết quả của GraphRAG chỉ cần chuyển về cùng format này.

In [ ]:
import time

from llama_index.core.llms import ChatMessage
from llama_index.core.vector_stores import FilterOperator, MetadataFilter, MetadataFilters

SYSTEM_PROMPT = """Bạn là trợ lý tư vấn bảo hiểm. Quy tắc bắt buộc:
1. Chỉ trả lời dựa trên các đoạn trích được cung cấp; không dùng kiến thức bên ngoài, không suy đoán.
2. Sau mỗi ý, ghi trích dẫn dạng [số] ứng với đoạn trích đã dùng, ví dụ: "... 1.000.000 đồng/ngày [2]".
3. Nêu đúng con số, điều kiện, giới hạn và tên sản phẩm/chương trình như trong tài liệu.
4. Nếu các đoạn trích không đủ để trả lời, nói rõ "Tôi không tìm thấy thông tin này trong tài liệu." và nêu phần còn thiếu.
5. Trả lời bằng tiếng Việt, ngắn gọn, có cấu trúc."""

USER_TEMPLATE = """Đoạn trích tài liệu:
{context}

Câu hỏi: {question}"""


def detect_products(question: str) -> list[str]:
    q = normalize(question)
    return sorted(
        code for code, aliases in PRODUCT_ALIASES.items()
        if any(re.search(rf"\b{re.escape(a)}\b", q) for a in aliases)
    )


def retrieve(question: str, top_k: int = TOP_K, products: list[str] | None = None) -> tuple[list[dict], list[str]]:
    """Trả về (sources, products). Câu hỏi có nhắc sản phẩm thì chỉ lấy tài liệu của sản phẩm đó và quy tắc chung."""
    if products is None:
        products = detect_products(question)
    filters = None
    if products:
        filters = MetadataFilters(filters=[
            MetadataFilter(key="product_code", value=[*products, "ALL"], operator=FilterOperator.IN),
        ])
    hits = index.as_retriever(similarity_top_k=top_k, filters=filters).retrieve(question)
    sources = [
        {
            "id": i,
            "file": hit.node.metadata.get("file_name"),
            "product": hit.node.metadata.get("product_code"),
            "section": hit.node.metadata.get("section", ""),
            "score": round(hit.score or 0.0, 3),
            "text": hit.node.get_content(),
        }
        for i, hit in enumerate(hits, start=1)
    ]
    return sources, products


def build_context(sources: list[dict]) -> str:
    return "\n\n".join(
        f"[{s['id']}] Nguồn: {s['file']} | Sản phẩm: {s['product']} | Mục: {s['section']}\n{s['text']}"
        for s in sources
    )


def parse_citations(text: str) -> list[int]:
    """Lấy các số trích dẫn trong câu trả lời: [1], [2, 3]..."""
    return sorted({int(x) for group in re.findall(r"\[([\d,\s]+)\]", text) for x in group.split(",") if x.strip().isdigit()})


def generate(question: str, sources: list[dict]) -> str:
    messages = [
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=USER_TEMPLATE.format(context=build_context(sources), question=question)),
    ]
    return (llm.chat(messages).message.content or "").strip()


def ask(question: str, top_k: int = TOP_K, products: list[str] | None = None) -> dict:
    t0 = time.perf_counter()
    sources, products = retrieve(question, top_k, products)
    t1 = time.perf_counter()
    answer = generate(question, sources)
    t2 = time.perf_counter()
    return {
        "question": question,
        "products": products,
        "answer": answer,
        "sources": sources,
        "cited": parse_citations(answer),
        "latency": {"retrieve": round(t1 - t0, 2), "generate": round(t2 - t1, 2), "total": round(t2 - t0, 2)},
    }


def show(result: dict) -> None:
    print("Câu hỏi :", result["question"])
    print("Sản phẩm:", result["products"] or "không lọc")
    print("\n" + result["answer"] + "\n")
    print("Nguồn (* = được trích dẫn):")
    for s in result["sources"]:
        mark = "*" if s["id"] in result["cited"] else " "
        print(f"  {mark}[{s['id']}] {s['score']:.3f}  {s['file']} › {s['section']}")
    print("Thời gian (s):", result["latency"])

### Thử với các câu hỏi ví dụ

Câu hỏi cụ thể là thế mạnh của RAG thông thường. Câu thứ hai được gõ không dấu để kiểm tra việc nhận diện sản phẩm.

In [ ]:
show(ask("Quyền lợi nằm viện của sản phẩm A là bao nhiêu?"))

In [ ]:
show(ask("San pham B tro cap nam vien bao nhieu mot ngay?"))

### Câu hỏi quan hệ: giới hạn của RAG thông thường

Để trả lời đủ câu này, cần ghép các điều kiện nằm rải rác ở nhiều Điều và nhiều file:

- định nghĩa Nằm viện (tối thiểu 24 giờ), thời gian chờ, điều kiện chung;
- loại trừ chung và loại trừ riêng của sản phẩm A;
- vi phạm nghĩa vụ kê khai, thời hạn nộp hồ sơ;
- điều kiện trợ cấp nằm viện của sản phẩm B.

Với top-k = 5, baseline thường chỉ lấy được một phần. Bạn có thể so sánh với `top_k=10`. Đây chính là loại câu hỏi GraphRAG sẽ xử lý ở giai đoạn 2.

In [ ]:
show(ask("Những điều kiện nào liên quan đến việc từ chối chi trả quyền lợi nằm viện?"))

## 7. Đánh giá nhanh

Mỗi câu hỏi mẫu đi kèm file nguồn mong đợi và từ khoá của đáp án. Các chỉ số:

| Chỉ số | Ý nghĩa |
|---|---|
| `retrieval_hit` | File đúng nằm trong top-k |
| `answer_hit` | Câu trả lời chứa ít nhất một từ khoá đáp án |
| `cited` | Câu trả lời có trích dẫn `[n]` |
| `latency_s` | Tổng thời gian truy xuất và sinh câu trả lời |

Câu cuối hỏi về "sản phẩm C", một sản phẩm không tồn tại. Câu này kiểm tra model có trả lời "không tìm thấy" thay vì bịa ra không.

Bộ câu hỏi này chỉ là điểm khởi đầu cho giai đoạn 4. Khi làm thật, nên thay bằng bộ câu hỏi do bộ phận nghiệp vụ soạn.

In [ ]:
import pandas as pd

EVAL_SET = [
    {"question": "Quyền lợi trợ cấp nằm viện của sản phẩm A chương trình Nâng cao là bao nhiêu một ngày?",
     "expect_file": "san_pham_A_an_tam_noi_tru.md", "expect_any": ["2.000.000", "2,000,000", "2 triệu"]},
    {"question": "Nằm viện ở ICU theo sản phẩm A được chi trả thế nào?",
     "expect_file": "san_pham_A_an_tam_noi_tru.md", "expect_any": ["200%", "gấp đôi"]},
    {"question": "Sản phẩm B trợ cấp nằm viện tối đa bao nhiêu ngày mỗi năm?",
     "expect_file": "san_pham_B_khoe_tron_doi.md", "expect_any": ["30 ngày"]},
    {"question": "Thời gian chờ đối với bệnh có sẵn là bao lâu?",
     "expect_file": "quy_tac_chung_suc_khoe.md", "expect_any": ["365 ngày", "1 năm"]},
    {"question": "Hồ sơ yêu cầu chi trả phải nộp trong thời hạn bao lâu?",
     "expect_file": "quy_tac_chung_suc_khoe.md", "expect_any": ["12 tháng"]},
    {"question": "Phí bảo hiểm của sản phẩm C là bao nhiêu?",
     "expect_file": None, "expect_any": ["không tìm thấy"]},
]


def evaluate(cases: list[dict]) -> pd.DataFrame:
    rows = []
    for case in cases:
        r = ask(case["question"])
        files = {s["file"] for s in r["sources"]}
        answer = normalize(r["answer"])
        rows.append({
            "question": case["question"],
            "retrieval_hit": None if case["expect_file"] is None else case["expect_file"] in files,
            "answer_hit": any(normalize(k) in answer for k in case["expect_any"]),
            "cited": bool(r["cited"]),
            "latency_s": r["latency"]["total"],
            "answer": r["answer"],
        })
    return pd.DataFrame(rows)


eval_df = evaluate(EVAL_SET)
display(eval_df.drop(columns="answer"))
print("retrieval_hit:", round(eval_df["retrieval_hit"].dropna().astype(float).mean(), 2))
print("answer_hit   :", round(eval_df["answer_hit"].mean(), 2))
print("cited        :", round(eval_df["cited"].mean(), 2))
print("latency TB   :", round(eval_df["latency_s"].mean(), 2), "s")

## Tiếp theo: giai đoạn 2

- Dùng cùng thư mục `data/docs/` để index bằng Microsoft GraphRAG, sinh ra entity, relationship và community report. Nên chạy `graphrag prompt-tune` để prompt trích xuất phù hợp với tiếng Việt và các loại entity bảo hiểm (sản phẩm, quyền lợi, điều kiện, loại trừ, thời gian chờ…).
- Chuyển kết quả GraphRAG Local/Global về cùng format `sources` ở trên, để tái dùng `generate()` và phần đánh giá.
- Các cải tiến có thể thử thêm cho baseline:
  - hybrid search (dense + sparse/BM25) cho mã sản phẩm và con số;
  - reranker, ví dụ `BAAI/bge-reranker-v2-m3`;
  - chuyển PDF thật sang Markdown, giữ nguyên cấu trúc Điều/Khoản.